# Sudoku AI — Exploration & Visualisation
**EMSI — Module Deep Learning 2025-2026**

Ce notebook couvre :
1. Génération et visualisation de puzzles Sudoku
2. Analyse du dataset d'entraînement
3. Comparaison des architectures après entraînement
4. Visualisation des courbes d'apprentissage
5. Analyse des erreurs et cas difficiles

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'backend'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import torch
import json
from pathlib import Path

plt.style.use('dark_background')
COLORS = ['#6366f1','#8b5cf6','#ec4899','#f59e0b','#10b981','#3b82f6']
MODEL_NAMES = ['mlp', 'cnn', 'rnn', 'lstm', 'gru', 'hybrid']
print('Setup OK')

## 1. Génération de puzzles Sudoku

In [ ]:
from utils.sudoku_utils import generate_sample, format_board, is_valid_solution

# Génèrer quelques puzzles
samples = [generate_sample(n) for n in [17, 25, 30, 36, 45]]
print(f'Généré {len(samples)} puzzles')
print(f'\nExemple (30 indices):')
print(format_board(samples[2][0]))

In [ ]:
def plot_sudoku(puzzle, solution=None, title='', ax=None, cmap_clue='#6366f1', cmap_sol='#10b981'):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4,4))
    board = np.array(puzzle).reshape(9,9)
    sol = np.array(solution).reshape(9,9) if solution is not None else board
    
    ax.set_xlim(0,9); ax.set_ylim(0,9)
    ax.set_aspect('equal')
    ax.axis('off')
    if title: ax.set_title(title, color='white', pad=8, fontsize=11)
    
    # Background
    ax.add_patch(patches.Rectangle((0,0),9,9, color='#0a0a1a'))
    
    # Cells
    for r in range(9):
        for c in range(9):
            val = board[r,c]
            disp = sol[8-r,c] if solution else (board[8-r,c] or '')
            orig = board[8-r,c]
            color = cmap_clue if orig != 0 else (cmap_sol if solution else '#888')
            if disp:
                ax.text(c+0.5, r+0.5, str(disp), ha='center', va='center',
                        fontsize=13, fontweight='bold', color=color)
    
    # Grid lines
    for i in range(10):
        lw = 2 if i % 3 == 0 else 0.5
        c = '#6366f1' if i % 3 == 0 else '#333'
        ax.axhline(i, color=c, lw=lw)
        ax.axvline(i, color=c, lw=lw)

# Afficher puzzles avec différents niveaux de difficulté
fig, axes = plt.subplots(1, 5, figsize=(20,4))
for i, (ax, (puz, sol)) in enumerate(zip(axes, samples)):
    clues = [17,25,30,36,45][i]
    plot_sudoku(puz, sol, f'{clues} indices', ax)

plt.suptitle('Puzzles Sudoku générés par backtracking', color='white', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('sudoku_examples.png', dpi=120, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

## 2. Analyse statistique du dataset

In [ ]:
from tqdm import tqdm
import random
from utils.sudoku_utils import generate_sample

N = 1000
clue_counts = []
digit_dist = np.zeros(10, dtype=int)

for _ in tqdm(range(N), desc='Génération'):
    n = random.randint(17, 45)
    puz, sol = generate_sample(n)
    clue_counts.append(n)
    for v in sol: digit_dist[v] += 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des indices
ax = axes[0]
ax.hist(clue_counts, bins=28, color='#6366f1', edgecolor='#4338ca', alpha=0.8)
ax.axvline(np.mean(clue_counts), color='#f59e0b', lw=2, label=f'Moyenne: {np.mean(clue_counts):.1f}')
ax.set_xlabel('Nombre d\'indices', color='white')
ax.set_ylabel('Fréquence', color='white')
ax.set_title('Distribution du nombre d\'indices', color='white')
ax.legend()
ax.tick_params(colors='white')
ax.set_facecolor('#12122a')

# Distribution des chiffres
ax = axes[1]
bars = ax.bar(range(1,10), digit_dist[1:], color=COLORS[:9], edgecolor='none', alpha=0.85)
ax.axhline(digit_dist[1:].mean(), color='white', lw=1.5, ls='--', label='Moyenne')
ax.set_xlabel('Chiffre', color='white')
ax.set_ylabel('Occurrences', color='white')
ax.set_title('Distribution des chiffres dans les solutions', color='white')
ax.legend()
ax.tick_params(colors='white')
ax.set_facecolor('#12122a')

plt.suptitle('Analyse statistique — 1000 puzzles', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('dataset_analysis.png', dpi=120, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()
print(f'Distribution chiffres: {dict(zip(range(1,10), digit_dist[1:]))}')

## 3. Courbes d'apprentissage — Tous les modèles

In [ ]:
WEIGHTS_DIR = Path('../backend/weights')

histories = {}
for name in MODEL_NAMES:
    hist_path = WEIGHTS_DIR / f'{name}_history.json'
    if hist_path.exists():
        with open(hist_path) as f:
            histories[name] = json.load(f)

if not histories:
    print('Aucun historique trouvé. Entraînez les modèles d\'abord :')
    print('  cd backend && python -m training.train_all --quick')
else:
    print(f'Historiques chargés: {list(histories.keys())}')

In [ ]:
if histories:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()
    
    for i, (name, h) in enumerate(histories.items()):
        ax = axes[i]
        color = COLORS[i % len(COLORS)]
        epochs = range(1, h['epochs_trained'] + 1)
        
        ax2 = ax.twinx()
        
        # Loss (left axis)
        ax.plot(epochs, h['train_loss'], color=color, lw=2, label='Train Loss', alpha=0.9)
        ax.plot(epochs, h['val_loss'],   color='#ef4444', lw=2, ls='--', label='Val Loss')
        ax.set_ylabel('Loss', color=color, fontsize=9)
        
        # Accuracy (right axis)
        ax2.plot(epochs, [v*100 for v in h['val_cell_acc']],   color='#10b981', lw=1.5, alpha=0.7, label='Cell Acc %')
        ax2.plot(epochs, [v*100 for v in h['val_puzzle_acc']], color='#f59e0b', lw=1.5, alpha=0.7, label='Puzzle Acc %')
        ax2.set_ylabel('Accuracy (%)', color='#10b981', fontsize=9)
        
        ax.set_title(f'{name.upper()} — {h["epochs_trained"]} époques', color='white', fontsize=11)
        ax.set_xlabel('Époque', color='white', fontsize=9)
        ax.set_facecolor('#12122a')
        ax.tick_params(colors='white')
        ax2.tick_params(colors='white')
        
        # Best puzzle acc
        best_puzz = max(h['val_puzzle_acc'])
        ax.text(0.98, 0.05, f'Best: {best_puzz*100:.1f}%', transform=ax.transAxes,
                ha='right', va='bottom', color='#f59e0b', fontsize=9, fontweight='bold')
        
        # Combined legend
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1+lines2, labels1+labels2, fontsize=7, loc='upper right')
    
    # Hide extra axes
    for j in range(len(histories), len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Courbes d\'apprentissage — Tous les modèles', color='white', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig('learning_curves.png', dpi=120, bbox_inches='tight', facecolor='#0a0a1a')
    plt.show()

## 4. Comparaison finale des performances

In [ ]:
summary_path = WEIGHTS_DIR / 'training_summary.json'
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    
    names, cell_accs, puzzle_accs, empty_accs = [], [], [], []
    for name, res in summary.items():
        m = res.get('test_metrics', {})
        if m:
            names.append(name.upper())
            cell_accs.append(m.get('cell_acc', 0) * 100)
            puzzle_accs.append(m.get('puzzle_acc', 0) * 100)
            empty_accs.append(m.get('empty_cell_acc', 0) * 100)
    
    x = np.arange(len(names))
    w = 0.28
    fig, ax = plt.subplots(figsize=(13, 6))
    
    bars1 = ax.bar(x - w, cell_accs,   w, label='Cell Accuracy (%)',       color='#6366f1', alpha=0.85)
    bars2 = ax.bar(x,     puzzle_accs, w, label='Puzzle Accuracy (%)',     color='#f59e0b', alpha=0.85)
    bars3 = ax.bar(x + w, empty_accs,  w, label='Empty Cell Accuracy (%)', color='#10b981', alpha=0.85)
    
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.1f}',
                   ha='center', va='bottom', fontsize=8, color='white')
    
    ax.set_xticks(x); ax.set_xticklabels(names, color='white', fontsize=11)
    ax.set_ylabel('Score (%)', color='white')
    ax.set_ylim(0, 115)
    ax.set_title('Comparaison des architectures — Test Set', color='white', fontsize=14)
    ax.legend(fontsize=10)
    ax.set_facecolor('#12122a')
    ax.tick_params(colors='white')
    ax.grid(axis='y', alpha=0.2)
    
    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight', facecolor='#0a0a1a')
    plt.show()
    
    # Table
    print(f'\n{"="*60}')
    print(f'{"Modèle":<10} {"CellAcc":>10} {"PuzzleAcc":>12} {"EmptyCellAcc":>14}')
    print(f'{"-"*60}')
    for n, c, p, e in zip(names, cell_accs, puzzle_accs, empty_accs):
        print(f'{n:<10} {c:>10.2f}% {p:>12.2f}% {e:>14.2f}%')
    print(f'{"="*60}')
else:
    print('Résumé non trouvé. Entraînez les modèles d\'abord.')

## 5. Analyse des erreurs sur un puzzle difficile

In [ ]:
import sys; sys.path.insert(0, '../backend')
from inference.solver import SudokuSolverEngine
from utils.sudoku_utils import generate_sample, is_valid_solution
import time

engine = SudokuSolverEngine()

# Puzzle très difficile (17 indices = minimum théorique)
puzzle, solution = generate_sample(17)
print(f'Puzzle avec {sum(1 for v in puzzle if v!=0)} indices (très difficile)')

results = engine.solve(puzzle.tolist())
print('\nRésultats:')
for name, r in results['results'].items():
    status = '✅' if r['is_valid'] else ('❌' if r['trained'] else '⬜')
    trained = '(entraîné)' if r['trained'] else '(non entraîné)'
    print(f'  {name:<15} {status} {r["time_ms"]:>8.2f}ms  conf={r["confidence"]:.3f}  {trained}')

In [ ]:
# Visualiser la solution de chaque modèle
trained_models = [(n, r) for n, r in results['results'].items() 
                  if r['trained'] and r['solution'] is not None]

ncols = min(len(trained_models) + 1, 4)
nrows = (len(trained_models) + 2) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows + 0.5))
if nrows == 1: axes = [axes]
axes_flat = [ax for row in axes for ax in (row if hasattr(row, '__iter__') else [row])]

# Original puzzle
plot_sudoku(puzzle, title=f'Puzzle ({sum(1 for v in puzzle if v!=0)} indices)', ax=axes_flat[0])

# Each model's solution
for i, (name, r) in enumerate(trained_models):
    color = r['color'] if 'color' in r else COLORS[i % len(COLORS)]
    title = f"{r['arch']} {'✅' if r['is_valid'] else '❌'}\n{r['time_ms']:.1f}ms"
    plot_sudoku(puzzle, r['solution'], title, axes_flat[i+1], cmap_sol=color)

for j in range(len(trained_models)+1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Analyse comparative — Puzzle difficile (17 indices)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()

## 6. Distribution des temps d'inférence

In [ ]:
N_BENCH = 50
bench_times = {name: [] for name in MODEL_NAMES + ['backtracking']}

for _ in range(N_BENCH):
    puz, _ = generate_sample(30)
    res = engine.solve(puz.tolist())
    for name, r in res['results'].items():
        if r['trained']:
            bench_times[name].append(r['time_ms'])

# Boxplot
fig, ax = plt.subplots(figsize=(12, 5))
valid_names = [n for n in bench_times if bench_times[n]]
data = [bench_times[n] for n in valid_names]
colors_bp = ['#6366f1','#8b5cf6','#ec4899','#f59e0b','#10b981','#3b82f6','#64748b'][:len(valid_names)]

bp = ax.boxplot(data, patch_artist=True, notch=False, 
                medianprops={'color': 'white', 'lw': 2})
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticks(range(1, len(valid_names)+1))
ax.set_xticklabels([n.upper() for n in valid_names], color='white')
ax.set_ylabel('Temps d\'inférence (ms)', color='white')
ax.set_title(f'Distribution des temps d\'inférence ({N_BENCH} puzzles)', color='white', fontsize=13)
ax.set_facecolor('#12122a')
ax.tick_params(colors='white')
ax.grid(axis='y', alpha=0.2)

# Means
for i, (name, vals) in enumerate(zip(valid_names, data)):
    ax.text(i+1, max(vals)*1.02, f'{np.mean(vals):.1f}ms', 
            ha='center', fontsize=8, color='white', alpha=0.7)

plt.tight_layout()
plt.savefig('inference_times.png', dpi=120, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()